In [ ]:
!pip install -U openai

In [ ]:
!apt-get update -y
!apt-get install -y texlive-xetex
!pip install pypandoc

import pypandoc
pypandoc.download_pandoc()

In [ ]:
import os
os.environ["HF_TOKEN"] = "" #Collect access token from Hugging Face
print(os.environ.get('HF_TOKEN'))

In [ ]:
import os
from openai import OpenAI
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
)

MODEL = "Qwen/Qwen3-VL-235B-A22B-Thinking:novita"

In [ ]:
FEATURE_FIELDS = [
    "mass_shape",
    "microcalcification_pattern",
    "architectural_distortion",
    "breast_density_category",
    "malignant"
]

tools = [
    {
        "type": "function",
        "name": "mammogram_feature_extraction",
        "description": "Extract categorical features from a mammogram for research use. Return image-level labels including mass shape, microcalcification pattern, architectural distortion, breast density category, and a binary malignancy decision.",
        "parameters": {
            "type": "object",
            "properties": {
                "mass_shape": {
                    "type": "string",
                    "enum": ["round", "oval", "lobulated", "irregular", "stellate"],
                    "description": "Shape of the detected mass. Lobulated is usually benign, irregular or stellate are suspicious.",
                },
                "microcalcification_pattern": {
                    "type": "string",
                    "enum": ["none", "scattered", "clustered", "linear", "segmental", "pleomorphic"],
                    "description": "Pattern of microcalcifications. Clustered, linear, segmental, and pleomorphic are highly suspicious.",
                },
                "architectural_distortion": {
                    "type": "string",
                    "enum": ["none", "mild", "moderate", "severe"],
                    "description": "Degree of tissue distortion. Moderate or severe distortion increases suspicion.",
                },
                "breast_density_category": {
                    "type": "string",
                    "enum": ["fatty", "scattered", "heterogeneously dense", "extremely dense"],
                    "description": "Shortened BI-RADS breast density categories with full text for heterogeneously dense.",
                },
                "malignant": {
                    "type": "string",
                    "enum": ["yes", "no"],
                    "description": "Binary label indicating whether the breast contains a malignant finding based on visible features in the image.",
                },
            },
            "required": [
                "mass_shape",
                "microcalcification_pattern",
                "architectural_distortion",
                "breast_density_category",
                "malignant",
            ],
        },
    }
]

SYSTEM_FEATURE_PROMPT = """
You are an expert breast radiologist specializing in mammography interpretation.
Your task is to analyze each mammogram image and extract a set of categorical features.
Provide strictly structured output using the defined function schema, with no free-text narrative.
For each image, return only the following standardized fields: mass_shape, microcalcification_pattern, architectural_distortion, breast_density_category, and malignant.
Be strictly objective and evidence-based; report only features visible on the provided image and avoid hallucinating any findings.
If a feature is not visible or cannot be determined from the image, mark it with the appropriate categorical value (e.g., 'none').
Do not speculate, infer, or incorporate information beyond what is directly observable in the mammogram.
Ensure consistency, reproducibility, and uniform use of BI-RADS terminology across all images analyzed.
Output must follow the function schema exactly, with all required fields present.
"""

FEATURE_PROMPT = """
Analyze the attached mammogram image.
Return a structured output ONLY, using the required categorical fields defined in the schema.

DO NOT provide any free text, explanations, or narrative descriptions.
Respond strictly in the format required by the function call and include ALL required fields.

Required output fields:
- mass_shape (round | oval | lobulated | irregular | stellate)
- microcalcification_pattern (none | scattered | clustered | linear | segmental | pleomorphic)
- architectural_distortion (none | mild | moderate | severe)
- breast_density_category (fatty | scattered | heterogeneously dense | extremely dense)
- malignant (yes | no)

Report only features that are directly visible in the provided mammogram image.
If a feature cannot be identified, assign the appropriate categorical value (e.g., "none").
Do not infer or speculate beyond the observable findings.
"""

REPORT_PROMPT = """
You are an expert breast radiologist dictating a complete, detailed mammography report.

You are given a structured set of imaging features extracted from a mammogram.
You MUST base your report ONLY on these features.
You are NOT allowed to introduce findings not directly supported by the input.

STRUCTURED FEATURES:
{features_json}

REPORT REQUIREMENTS:

- Examination: Specify mammography examination (screening mammography).
- Breast Composition: Translate breast_density_category into standard BI-RADS language.
- Findings: Describe mass morphology, calcification pattern, and architectural distortion.
  Do not invent locations, sizes, or laterality.
- Impression: Summarize overall concern level using professional radiology tone.
- BI-RADS Assessment: Assign a BI-RADS category consistent with the provided features.
- Recommendation: Provide an appropriate next step (routine screening, short-term follow-up, or diagnostic workup).

STYLE RULES:
- Clinical, neutral, professional tone
- No disclaimers
- No speculation
- No future prediction beyond standard radiology practice
- Bullet points where appropriate
"""

In [ ]:
import os
import json
import datetime
from pathlib import Path
from PIL import Image
import io, base64
import pypandoc


OUTPUT_DIR = "generated_files/report"
os.makedirs(OUTPUT_DIR, exist_ok=True)



def encode_image(image_path):
    img = Image.open(image_path)
    img.thumbnail((1024, 1024), Image.Resampling.LANCZOS)

    buffer = io.BytesIO()
    img.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")


def extract_features(image_path: str) -> dict:
    img = encode_image(image_path)

    PROMPT = "Analyze this mammogram image."

    response = client.chat.completions.create(
        model=MODEL,
        messages = [
            {
                "role": "system",
                "content": SYSTEM_FEATURE_PROMPT
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": FEATURE_PROMPT},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{img}"
                        }
                    }
                ]
            }
        ],
        tools=tools,  
        tool_choice={
            "type": "function",
            "function": {"name": "mammogram_feature_extraction"}
        },
        temperature=0.0,
        top_p=0.1,
        extra_body={"top_k": 0},
    )

    tool_call = response.choices[0].message.tool_calls[0]
    features = json.loads(tool_call.function.arguments)  
    return features

def generate_report_from_features(features: dict) -> str:
    SYSTEM_PROMPT = REPORT_PROMPT.format(features_json=json.dumps(features, indent=2))
    PROMPT = "Generate the mammography report."
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": PROMPT}
        ],
        temperature=0.0,
        top_p=0.1,
        extra_body={"top_k": 0},
    )
    report_text = response.choices[0].message.content
    return report_text.strip()


def generate_pdf(run_id, report_text, metadata):
    report_md = f"""
# Mammogram Report

**Run ID:** {metadata['run_id']}  
**Model:** {metadata['model']}  
**Date:** {metadata['timestamp']}  

---

{report_text}

---

## Notes
- For research use only
"""
    pdf_path = os.path.join(OUTPUT_DIR, f"{run_id}_mammogram_report.pdf")
    try:
        pypandoc.convert_text(report_md, "pdf", format="md", outputfile=pdf_path, extra_args=["--pdf-engine=xelatex"])
        return pdf_path
    except RuntimeError:
        fallback = os.path.join(OUTPUT_DIR, f"{run_id}_report.md")
        with open(fallback, "w", encoding="utf-8") as f:
            f.write(report_md)
        return fallback


def run_pipeline(image_path: str):
    run_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    metadata = {
        "run_id": run_id,
        "image_id": Path(image_path).stem,
        "model": MODEL,
        "timestamp": datetime.datetime.now().isoformat(),
    }

    features = extract_features(image_path)
    print("Extracted features:")
    print(json.dumps(features, indent=2))

    report = generate_report_from_features(features)
    pdf_path = generate_pdf(run_id, report, metadata)
    print(f"Report saved to: {pdf_path}")


if __name__ == "__main__":
    os.system("rm -rf generated_files")
    IMAGE_PATH = "" #Set location to a single png file in the dataset folder
    run_pipeline(IMAGE_PATH)